In [2]:
import pandas as pd
import numpy as np

movies = pd.read_csv("../../data/movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
# Keep genres as separate tokens. "Sci-Fi" and "Film-Noir" contain hyphens,
# so we split on "|" ourselves instead of letting sklearn split on punctuation.
movies["genre_tokens"] = movies["genres"].replace("(no genres listed)", "")

movies[["title", "genre_tokens"]].head()

,title,genre_tokens
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance
4,Father of the Bride Part II (1995),Comedy


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(tokenizer=lambda x: x.split("|"), token_pattern=None, lowercase=False)
tfidf_matrix = tfidf.fit_transform(movies["genre_tokens"])

print("Shape:", tfidf_matrix.shape)
print(tfidf.get_feature_names_out())

Shape: (9742, 20)
['' 'Action' 'Adventure' 'Animation' 'Children' 'Comedy' 'Crime'
 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'IMAX' 'Musical'
 'Mystery' 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western']


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Similarity of every movie to every other movie
sim = cosine_similarity(tfidf_matrix)
print(sim.shape)   # (9742, 9742)

(9742, 9742)


In [6]:
title_to_idx = pd.Series(movies.index, index=movies["title"])

def similar_movies(title, n=10):
    idx = title_to_idx[title]
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()

    # sort high to low, skip the movie itself (first one)
    top_idx = scores.argsort()[::-1][1:n + 1]

    result = movies.iloc[top_idx][["title", "genres"]].copy()
    result["similarity"] = scores[top_idx].round(3)
    return result

similar_movies("Toy Story (1995)")

,title,genres,similarity
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
8927,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,1.0
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
